# 02 -- Logistic Regression on a9a (Binary Classification)

**a9a** is a binarized version of the UCI Adult / census-income dataset: predict
whether a person's income exceeds \$50K/year from 123 sparse demographic features.
~32.5K training rows, 16.3K test rows.

We fit `models.LogisticRegression` -- binary cross-entropy + L2 regularization,
trained with every optimizer in the library -- and compare them on loss,
accuracy, gradient norm, runtime, and memory.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
%matplotlib inline

from datasets.loaders import load_a9a
from models import LogisticRegression
from optimizers import build_optimizer
from experiments.compare_optimizers import compare_optimizers
from visualization.plots import (
    plot_loss_curves, plot_accuracy_curves, plot_gradient_norms,
    plot_comparison_bars, plot_confusion_matrix, plot_runtime_vs_loss,
)

ds = load_a9a()
print(ds)

## Loss and gradient

$$\mathcal L(w) = -\frac{1}{n}\sum_i \big[y_i \log\sigma(w^\top x_i) + (1-y_i)\log(1-\sigma(w^\top x_i))\big] + \frac{\lambda}{2}\lVert w\rVert_2^2$$
$$\nabla \mathcal L(w) = \frac1n X^\top(\sigma(Xw) - y) + \lambda w$$

labels are accepted as either $\{0,1\}$ or $\{-1,+1\}$ (a9a ships $\{-1,+1\}$) and normalized internally.

In [ ]:
optimizer_lrs = {
    'Gradient Descent': 0.5,
    'SGD': 0.05,
    'Adagrad': 0.5,
    'AdaGrad-Norm': 1.0,
    'RMSProp': 0.01,
    'Adam': 0.01,
    'L-BFGS': 1.0,
}
optimizers = {name: build_optimizer(name, lr=lr) for name, lr in optimizer_lrs.items()}

result = compare_optimizers(
    lambda: LogisticRegression(l2=1e-4),
    optimizers,
    ds.X_train, ds.y_train, ds.X_test, ds.y_test,
    epochs=60, batch_size=256, track_memory=True,
)
result.table

In [ ]:
plot_loss_curves(result.histories); plt.show()
plot_accuracy_curves(result.histories); plt.show()
plot_gradient_norms(result.histories); plt.show()
plot_runtime_vs_loss(result.histories); plt.show()

In [ ]:
plot_comparison_bars(result.table, 'runtime_sec', 'seconds'); plt.show()
plot_comparison_bars(result.table, 'test_accuracy', 'accuracy'); plt.show()
plot_comparison_bars(result.table, 'peak_memory_mb', 'MB'); plt.show()

## Confusion matrix for the best optimizer

In [ ]:
best_name = result.table['test_accuracy'].astype(float).idxmax()
best_result = result.results[best_name]
print('Best optimizer:', best_name)

model = LogisticRegression(l2=1e-4)
Xb_test = model._add_intercept(ds.X_test)
preds = model.predict(best_result.final_params, Xb_test)
y_true = model._normalize_labels(ds.y_test)
plot_confusion_matrix(y_true, preds, class_names=['<=50K', '>50K'], title=f'a9a -- {best_name}')
plt.show()

Continue to **`03_softmax_regression_mnist.ipynb`** for the multi-class case.